# 08 — Premier entraînement sur le dataset Luna revu

Objectif : mesurer ce que le **nouveau dataset seul** apporte au nano, sans ajouter les images d’Infactory à l’entraînement.

Nous comparons le nano COCO préentraîné, notre ancien nano v002 (10 époques sur Infactory) et le nano entraîné ici. Même résolution 640, mêmes images de test, mêmes seuils.

C’est un pilote privé : les droits de redistribution de la vidéo ne sont pas établis. Ce notebook ne publie rien et ne lance aucun appel Luna payant.

In [ ]:
import csv
import gc
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("YOLO_AUTOINSTALL", "false")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_ROOT = PROJECT_ROOT / "data/processed/luna_v001"
RUN_ROOT = PROJECT_ROOT / "models/checkpoints/baseline_runs/yolo26n_luna_v001_20260928_pilot"
REPORT_PATH = RUN_ROOT / "report.json"
build = json.loads((DATA_ROOT / "build_report.json").read_text())
with (DATA_ROOT / "manifest.csv").open() as stream:
    manifest = list(csv.DictReader(stream))
print("Projet :", PROJECT_ROOT)
print("Résultats :", RUN_ROOT)

## 1. Ce qui entre dans l’expérience

La compilation contient plusieurs matchs. Nous avons retrouvé **10 groupes probables** en inspectant les scores, les maillots et les terrains. Tous les extraits et ralentis d’un même groupe restent dans un seul split. Ce sont des identités inférées visuellement, pas des identifiants de matchs vérifiés.

Les répartitions possibles avec 6 groupes train, 2 validation et 2 test sont comparées pour approcher 70/15/15 en images, positifs et négatifs. Aucune prédiction du modèle n’intervient dans ce choix.

Aux neuf transitions entre matchs, une petite marge retire les images fondues mélangeant deux matchs. Les décisions humaines originales restent inchangées.

In [ ]:
lines = ["| Split | Images | Avec ballon | Sans ballon | Groupes de matchs |",
         "|---|---:|---:|---:|---|"]
for split, row in build["splits"].items():
    lines.append(f"| {split} | {row['images']} | {row['positive']} | {row['negative']} | {', '.join(row['matches'])} |")
display(Markdown("\n".join(lines)))
print("Exclusions :", build["excluded"])
print("Images acceptées sans réponse Luna :", build["images_without_luna_answer_submitted_empty"])
print("Stockage :", build["image_storage"])
# Vérification de l'exclusivité des groupes.
owners = {}
for row in manifest:
    owners.setdefault(row["match_id"], set()).add(row["split"])
assert all(len(splits) == 1 for splits in owners.values())
assert len(manifest) == build["images"]
print("Aucun groupe partagé entre les splits.")

**Attention à la qualité des négatifs :** 49 images conservées n’avaient pas de réponse Luna et ont été soumises vides lors de l’acceptation globale demandée. Une boîte vide n’est donc pas une preuve de l’absence du ballon. Cela peut biaiser les métriques ; ces cas restent traçables dans le manifeste.

Les JPEG sont liés aux images déjà présentes, sans recopier leurs données. Ne pas modifier ces fichiers en place.

In [ ]:
overview = PROJECT_ROOT / "models/reports/luna_scene_overview.jpg"
if overview.exists():
    display(DisplayImage(filename=str(overview), width=900))

## 2. Paramètres et lancement

Pour ce pilote : GPU Apple MPS, batch 2, aucune mise en cache des images, 640 px, AdamW. Initialisation avec le nano COCO, **pas avec le checkpoint Infactory**.

Le premier lancement utilise une limite temporelle de 18 minutes et un arrêt anticipé après 5 époques sans amélioration. Dans Ultralytics, une limite temporelle peut recalculer le nombre d’époques : le nombre réellement terminé est affiché plus bas.

La cellule suivante est désactivée par défaut pour ne pas relancer le travail par accident. Pour une nouvelle expérience, changer RUN_ROOT vers un nom inédit et activer RUN_TRAINING.

In [ ]:
RUN_TRAINING = False
EPOCHS = 15
IMAGE_SIZE = 640
BATCH = 2
DEVICE = "mps"

if RUN_TRAINING:
    import subprocess
    import time
    if RUN_ROOT.exists():
        raise FileExistsError("Choisir un RUN_ROOT inédit : aucun résultat existant ne sera remplacé.")
    log_path = PROJECT_ROOT / "models/reports" / (RUN_ROOT.name + ".log")
    command = [sys.executable, "-u", str(PROJECT_ROOT / "scripts/train_luna_baseline.py"),
               "--epochs", str(EPOCHS), "--imgsz", str(IMAGE_SIZE),
               "--batch", str(BATCH), "--device", DEVICE, "--output", str(RUN_ROOT)]
    status = display(Markdown("Démarrage…"), display_id=True)
    with log_path.open("w") as log:
        process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            tail = log_path.read_text(errors="replace").splitlines()[-5:]
            status.update(Markdown("~~~text\n" + "\n".join(tail) + "\n~~~"))
            time.sleep(2)
    if process.returncode:
        raise RuntimeError(f"Échec : consulter {log_path}")
    status.update(Markdown("Entraînement et évaluations terminés."))

In [ ]:
report = json.loads(REPORT_PATH.read_text())
display(Markdown(f"**État :** {report['status']} — **Époques enregistrées :** {report.get('epochs_logged', report.get('epochs_completed', 'en cours'))}"))
print("La limite temporelle peut arrêter la dernière époque au milieu du passage sur les images.")
if report["status"] != "complete":
    print("Le rapport est partiel ; réexécuter cette cellule après la fin.")
print("Configuration :", report["training"])

## 3. Comparaison à conditions identiques

La validation sert à choisir best.pt. Le test est utilisé **seulement après l’entraînement et ce choix**, pour comparer les trois modèles.

Précision, rappel, F1 et TP/FP/FN sont calculés avec confiance ≥ 0,25 et IoU ≥ 0,50. Un TP est une boîte appariée à une annotation ; un FP est une boîte non appariée ; un FN est une annotation manquée.

Le mAP utilise toute la courbe précision/rappel, pas seulement le seuil 0,25. mAP50–95 demande aussi des boîtes plus précisément localisées. Pour le modèle COCO, seule la classe « sports ball » est comparée aux annotations « ball ». Pas de comparaison entre les métriques d’anciens et nouveaux splits.

In [ ]:
names = {"pretrained": "Nano COCO", "v002": "Ancien nano Infactory", "luna": "Nouveau nano Luna"}
def metrics_table(results):
    lines = ["| Modèle | mAP50 | mAP50–95 | Précision | Rappel | F1 | TP | FP | FN | FP sur négatifs |",
             "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for name, row in results.items():
        fixed = row["fixed"]
        values = [row["map50"], row["map50_95"], fixed["precision"], fixed["recall"], fixed["f1"]]
        formatted = " | ".join(f"{v:.1%}" for v in values)
        counts = " | ".join(str(fixed.get(k, 0)) for k in ["tp", "fp", "fn", "fp_on_negatives"])
        lines.append(f"| {names[name]} | {formatted} | {counts} |")
    display(Markdown("\n".join(lines)))

display(Markdown("### Validation — 176 images"))
metrics_table(report["validation"])
display(Markdown("### Test final — 176 images de deux autres matchs"))
metrics_table(report["test"])

## 4. Courbes d’apprentissage

Une baisse des pertes train accompagnée d’une hausse durable des pertes validation et d’une dégradation des métriques peut signaler du surapprentissage. Une seule remontée n’est pas suffisante pour conclure.

Les valeurs train/validation ne sont pas parfaitement comparables : augmentations et conditions de calcul diffèrent. Regarder les tendances sur plusieurs époques et les performances validation ensemble.

In [ ]:
with (RUN_ROOT / "train/results.csv").open() as stream:
    history = [{k.strip(): float(v) for k, v in row.items()} for row in csv.DictReader(stream)]
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, loss in zip(axes.flat[:3], ["box_loss", "cls_loss", "l1_loss"]):
    for split in ["train", "val"]:
        key = f"{split}/{loss}"
        if key in history[0]:
            ax.plot(epochs, [row[key] for row in history], marker=".", label=split)
    ax.set(title=loss, xlabel="Époque", ylabel="Perte")
    ax.legend()
    ax.grid(alpha=.25)
ax = axes.flat[3]
for key, label in [("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50–95")]:
    ax.plot(epochs, [row[key] for row in history], marker=".", label=label)
ax.set(title="Validation", xlabel="Époque", ylabel="mAP", ylim=(0, 1))
ax.legend()
ax.grid(alpha=.25)
fig.suptitle("Apprentissage sur le nouveau dataset uniquement")
fig.tight_layout()
curves_path = RUN_ROOT / "learning_curves.png"
fig.savefig(curves_path, dpi=120)
plt.close(fig)
display(DisplayImage(filename=str(curves_path)))
display(Markdown(f"{len(history)} époques observées. Ce pilote court ne permet pas de conclure à lui seul sur le surapprentissage."))

## 5. Matrices de confusion au seuil utilisé

Pour une détection à une classe, la matrice n’est pas celle d’une classification classique. La case « fond/fond » n’a pas de nombre naturel de vrais négatifs : il existe une infinité de zones sans ballon.

Ici les lignes sont les prédictions, les colonnes les annotations. Les valeurs viennent exactement des TP/FP/FN du tableau à confiance 0,25 et IoU 0,50. Les matrices automatiques d’Ultralytics peuvent utiliser un autre seuil ; elles ne sont pas utilisées ci-dessous.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.7))
for ax, (name, row) in zip(axes, report["test"].items()):
    fixed = row["fixed"]
    counts = np.array([[fixed.get("tp", 0), fixed.get("fp", 0)], [fixed.get("fn", 0), np.nan]])
    ax.imshow(np.nan_to_num(counts), cmap="Blues", vmin=0, vmax=138)
    for i, j in np.ndindex(2, 2):
        ax.text(j, i, "N/A" if np.isnan(counts[i, j]) else str(int(counts[i, j])),
                ha="center", va="center", color="black", fontsize=13)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Ballon", "Fond"],
           yticklabels=["Ballon", "Fond"], xlabel="Annotation", ylabel="Prédiction", title=names[name])
fig.tight_layout()
matrix_path = RUN_ROOT / "fixed_confusion_matrices.png"
fig.savefig(matrix_path, dpi=120)
plt.close(fig)
display(DisplayImage(filename=str(matrix_path)))

## 6. Courbes précision/rappel du nouveau modèle

La courbe PR fait varier le seuil de confiance ; la courbe F1 montre l’effet de ce seuil sur le compromis précision/rappel. Le test reste un compte rendu, pas un outil pour régler le modèle à répétition.

In [ ]:
for filename in ["BoxPR_curve.png", "BoxF1_curve.png"]:
    path = RUN_ROOT / "evaluation/test_luna" / filename
    if path.exists():
        display(DisplayImage(filename=str(path), width=800))

## 7. Inspection avant/après, sans surcharger la mémoire

Avant = ancien nano Infactory ; après = nouveau nano entraîné sur Luna. Le contour **vert** est l’annotation revue, le **rouge** la prédiction à confiance ≥ 0,25. Le texte est placé à l’extérieur des boîtes.

Les 176 images de test sont traitées une par une sur CPU, puis on ne conserve que les coordonnées. La galerie montre au maximum trois exemples par catégorie, avec des matchs diversifiés quand possible. Aucune copie des photos n’est créée.

« Corrigée » : erreurs avant, aucune après. « Régression » : correcte avant, erreurs après. « Encore incorrecte » peut aussi représenter une localisation trop imprécise, même si le ballon est visuellement trouvé.

In [ ]:
RUN_VISUAL_INSPECTION = True
ERROR_PATH = RUN_ROOT / "error_analysis_validation.json"

if RUN_VISUAL_INSPECTION and report["status"] == "complete" and not ERROR_PATH.exists():
    import torch
    from ultralytics import YOLO
    from ultralytics.utils import ops

    from pitchlens.training.diagnostics import (
        change_category,
        quiet_ultralytics_progress,
    )
    from pitchlens.training.evaluate import BallValidator

    class GalleryValidator(BallValidator):
        """Capture the exact predictions used by the metric table, one frame at a time."""
        def init_metrics(self, model):
            super().init_metrics(model)
            self.gallery = {}

        def update_metrics(self, preds, batch):
            for index, pred in enumerate(preds):
                target = self._prepare_batch(index, batch)
                selected = pred["conf"] >= .25
                filtered = {key: value[selected] for key, value in pred.items()}
                tp = int(self._process_batch(filtered, target)["tp"][:, 0].sum())
                boxes = ops.scale_boxes(target["imgsz"], filtered["bboxes"].clone(),
                                        target["ori_shape"], target["ratio_pad"]).cpu().numpy()
                h, w = target["ori_shape"]
                self.gallery[Path(target["im_file"]).stem] = {
                    "boxes": (boxes / np.array([w, h, w, h])).tolist(),
                    "confidence": filtered["conf"].cpu().tolist(),
                    "counts": {"tp": tp, "fp": len(boxes)-tp, "fn": len(target["cls"])-tp},
                }
            super().update_metrics(preds, batch)

        def finalize_metrics(self):
            super().finalize_metrics()
            self.metrics.gallery = self.gallery

    torch.set_num_threads(4)
    test_rows = [row for row in manifest if row["split"] == "test"]
    analysis = []
    for row in test_rows:
        label_path = DATA_ROOT / "test/labels" / (row["item_id"] + ".txt")
        ground_truth = []
        for line in label_path.read_text().splitlines():
            cls, x, y, w, h = map(float, line.split())
            ground_truth.append([x-w/2, y-h/2, x+w/2, y+h/2])
        analysis.append({"item_id": row["item_id"], "match_id": row["match_id"],
                         "ground_truth": ground_truth})
    for key, label, weight in [("before", "v002", report["previous_weights"]),
                               ("after", "luna", report["best_weights"])]:
        model = YOLO(weight)
        with quiet_ultralytics_progress():
            metrics = model.val(validator=GalleryValidator, data=str(DATA_ROOT / "dataset.yaml"),
                                split="test", imgsz=640, device="cpu", batch=1, workers=0,
                                conf=.001, iou=.7, max_det=100, plots=False, verbose=False,
                                project=str(RUN_ROOT / "evaluation"), name="gallery_"+label, exist_ok=True)
        for count in ["tp", "fp", "fn"]:
            assert metrics.fixed_threshold[count] == report["test"][label]["fixed"][count]
        for row in analysis:
            prediction = metrics.gallery[row["item_id"]]
            row[key] = prediction["boxes"]
            row[key+"_confidence"] = prediction["confidence"]
            row[key+"_counts"] = prediction["counts"]
        del model, metrics
        gc.collect()
    for row in analysis:
        row["category"] = change_category(row["before_counts"], row["after_counts"])
    ERROR_PATH.write_text(json.dumps(analysis, indent=2))
    print("Inspection calculée : les totaux correspondent exactement au tableau.")
elif ERROR_PATH.exists():
    print("Inspection déjà calculée : aucune inférence relancée.")

In [ ]:
if ERROR_PATH.exists():
    from collections import Counter

    from PIL import Image as PILImage
    analysis = json.loads(ERROR_PATH.read_text())
    print("Catégories sur les 176 images :", dict(Counter(row["category"] for row in analysis)))
    lines = ["| Match de test | Modèle | Images | TP | FP | FN | F1 |",
             "|---|---|---:|---:|---:|---:|---:|"]
    for match in sorted({row["match_id"] for row in analysis}):
        rows = [row for row in analysis if row["match_id"] == match]
        for key, label in [("before", "Ancien"), ("after", "Nouveau")]:
            counts = Counter()
            for row in rows:
                counts.update(row[key + "_counts"])
            tp, fp, fn = (counts[k] for k in ["tp", "fp", "fn"])
            f1 = 2*tp / (2*tp+fp+fn) if 2*tp+fp+fn else 0
            lines.append(f"| {match} | {label} | {len(rows)} | {tp} | {fp} | {fn} | {f1:.1%} |")
    display(Markdown("\n".join(lines)))

    def diverse_examples(rows, limit=3):
        chosen = []
        for match in sorted({row["match_id"] for row in rows}):
            chosen.append(next(row for row in rows if row["match_id"] == match))
        for row in rows:
            if len(chosen) >= limit:
                break
            if row not in chosen:
                chosen.append(row)
        return chosen[:limit]

    for category, title in [("corrected", "Erreurs corrigées"),
                            ("regressed", "Régressions"),
                            ("still_wrong", "Erreurs restantes")]:
        selected = diverse_examples([r for r in analysis if r["category"] == category])
        if not selected:
            display(Markdown(f"### {title}\nAucune dans ce test au seuil choisi."))
            continue
        fig, axes = plt.subplots(len(selected), 2, figsize=(12, 3.5*len(selected)), squeeze=False)
        for i, row in enumerate(selected):
            with PILImage.open(DATA_ROOT / "test/images" / (row["item_id"] + ".jpg")) as original:
                original.thumbnail((800, 450))
                pixels = np.asarray(original).copy()
            h, w = pixels.shape[:2]
            for j, (key, label) in enumerate([("before", "Avant — Infactory"), ("after", "Après — Luna")]):
                ax = axes[i, j]
                ax.imshow(pixels)
                for box in row["ground_truth"]:
                    x1, y1, x2, y2 = np.array(box)*[w, h, w, h]
                    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=1))
                for box, confidence in zip(row[key], row[key+"_confidence"]):
                    x1, y1, x2, y2 = np.array(box)*[w, h, w, h]
                    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=.8))
                    ax.text(x1, max(9, y1-5), f"{confidence:.0%}", fontsize=7, color="red",
                            bbox={"facecolor": "white", "alpha": .8, "pad": .5, "edgecolor": "none"})
                counts = row[key+"_counts"]
                ax.set_title(f"{label} · {row['match_id']} · TP {counts['tp']} / FP {counts['fp']} / FN {counts['fn']}", fontsize=9)
                ax.axis("off")
        fig.suptitle(title + " — vert : annotation, rouge : prédiction")
        fig.tight_layout()
        gallery_path = RUN_ROOT / (category + "_gallery.png")
        fig.savefig(gallery_path, dpi=110)
        plt.close(fig)
        display(DisplayImage(filename=str(gallery_path)))

## 8. Benchmark de décision caméra : une seule cible

Sur les **mêmes 176 images de test**, appliquer une règle proche de l’usage prévu :

1. Rejeter les propositions sous le seuil de confiance (25 % par défaut).
2. Garder uniquement la boîte la plus confiante ; en cas d’égalité, la première.
3. Accepter cette cible si son IoU avec un ballon annoté atteint 10 % ou 20 %.

IoU = intersection / union : « 20 % » veut dire IoU ≥ 0,20, pas 20 % de la surface d’une boîte. Une boîte secondaire fausse ne pénalise plus la décision si la meilleure vise le ballon. On ne choisit jamais la bonne boîte en consultant l’annotation.

Précision = bonnes cibles / cibles sélectionnées. Rappel = bonnes cibles / images contenant un ballon annoté. Choisir une mauvaise cible sur une image positive compte comme FP + FN, mais une seule mauvaise décision. Sur une image vide, rester silencieux est correct ; sélectionner une cible est une fausse alerte.

**Limites :** ce n’est pas encore un test de stabilité temporelle, de latence ou du gimbal. Avec plusieurs vrais ballons, les labels ne disent pas lequel est en jeu : toute boîte annotée peut compter comme correcte. Parmi les 38 négatifs du test, 12 viennent d’une réponse Luna manquante acceptée vide : les alertes restent à revoir visuellement.

Le seuil est fixé avant la lecture du test. Pour le régler, utiliser la validation et garder un autre test indépendant. Le cache ne contient que les propositions ≥ 25 % : un seuil inférieur nécessite une nouvelle inférence.

In [ ]:
import hashlib

from pitchlens.training.camera_benchmark import benchmark_targets

CAMERA_CONFIDENCE = .25
CAMERA_IOUS = [.10, .20]
camera_rows = json.loads(ERROR_PATH.read_text())
camera_report = {
    "scope": "offline frame-level camera decision proxy, not temporal tracking",
    "confidence": CAMERA_CONFIDENCE,
    "selection": "highest confidence only; first wins exact ties",
    "matching": "IoU with any annotated ball; active ball not identified",
    "cached_min_confidence": .25,
    "cache_sha256": hashlib.sha256(ERROR_PATH.read_bytes()).hexdigest(),
    "test_note": "Two inferred matches; 12 empty labels without a Luna response",
    "results": {},
}
lines = ["| Modèle | IoU min. | Bonnes cibles | Précision | Rappel | F1 | Mauvaises cibles sur positifs | Alertes sur négatifs |",
         "|---|---:|---:|---:|---:|---:|---:|---:|"]
for iou in CAMERA_IOUS:
    for key, label in [("before", "Ancien Infactory"), ("after", "Nouveau Luna")]:
        result = benchmark_targets(camera_rows, key, confidence=CAMERA_CONFIDENCE, matching_iou=iou)
        camera_report["results"][f"{key}_iou{iou:.2f}"] = result
        metrics = result["metrics"]
        lines.append(
            f"| {label} | {iou:.0%} | {metrics['tp']}/{metrics['positive_images']} | "
            f"{metrics['precision']:.1%} | {metrics['recall']:.1%} | {metrics['f1']:.1%} | "
            f"{metrics.get('wrong_target', 0)} | {metrics.get('false_alarm', 0)}/{metrics['negative_images']} |"
        )
display(Markdown("\n".join(lines)))
(RUN_ROOT / "camera_benchmark.json").write_text(json.dumps(camera_report, ensure_ascii=False, indent=2))
display(Markdown(
    "À confiance 25 %, les IoU 10 % et 20 % donnent ici les mêmes résultats. "
    "Le nouveau modèle n'est pas meilleur selon cette règle opérationnelle : "
    "les métriques classiques et la décision caméra mesurent des choses différentes."
))

## 9. Lecture du premier pilote

Sur les deux matchs de test, le nouveau nano passe de **14,9 % à 33,6 % de mAP50** par rapport à notre ancien modèle Infactory. Son rappel passe de **18,8 % à 37,0 %**, mais les faux positifs augmentent de **60 à 134** au seuil 0,25. La précision baisse de 30,2 % à 27,6 %. Le F1 progresse de 23,2 % à 31,6 %.

Le nano COCO sans adaptation obtient 24,8 % de mAP50 et 29,1 % de F1 : le nouveau modèle améliore ces scores, mais reste beaucoup moins sélectif au seuil fixé. Il ne faut donc pas conclure qu’il est déjà plus fiable pour piloter la caméra.

Le meilleur checkpoint choisi en validation atteint 53,0 % de mAP50 sur ce split, contre 33,6 % sur le test : les deux autres matchs restent difficiles. Cet écart, sur si peu de matchs, ne prouve pas à lui seul un surapprentissage.

Ce test mesure un changement de dataset **et** d’entraînement, pas uniquement un changement de modèle. Les annotations, la diversité des matchs et la quantité de données peuvent toutes influencer les résultats. Quinze époques complètes et une dernière partielle ont été enregistrées.

**Suite conseillée :** contrôler les fausses détections et les négatifs sans réponse Luna, puis ajouter d’autres matchs avec des identifiants explicites. Si l’on ajuste le seuil de confiance, le choisir sur la validation, pas sur le test. Garder une nouvelle évaluation indépendante pour les prochains choix.